# PoC de búsqueda de momentos con LanguageBind

Este notebook carga o construye embeddings por ventanas para un vídeo de gameplay, y luego permite repetir consultas de texto sin volver a codificar vídeo/audio.


## Imports Y Rutas

Define las rutas del repo, recarga los módulos locales para que el notebook recoja cambios recientes, e importa los helpers que se usan después.


In [1]:
from pathlib import Path
import importlib
import sys
import subprocess

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data" / "proxy_720p30.mp4").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

# Jupyter mantiene módulos importados entre re-ejecuciones. Los quitamos
# para que el notebook recoja cambios en scripts/tfvtg_poc.py al momento.
sys.modules.pop("scripts.tfvtg_poc", None)
sys.modules.pop("scripts.build_languagebind_index", None)
importlib.invalidate_caches()

from scripts.tfvtg_poc import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_AUDIO_WEIGHT,
    DEFAULT_CLIP_CONTEXT_SECONDS,
    DEFAULT_NUM_WORKERS,
    DEFAULT_STRIDE_SECONDS,
    DEFAULT_WINDOW_SECONDS,
    DEFAULT_VIDEO_WEIGHT,
    export_result_clips,
    extract_scene_embeddings_batched,
    extract_text_embedding,
    load_audio_track,
    load_languagebind_models,
    make_windows,
    open_video_reader,
    print_results,
    rank_windows,
    weighted_fuse_embeddings,
    require_cuda,
)
from scripts.build_languagebind_index import (
    build_languagebind_index,
    default_index_dir,
    load_languagebind_index,
)


## Configuración

Los defaults siguen centralizados en los scripts. Los comentarios inline muestran el valor actual que usa este notebook.


In [34]:
VIDEO_PATH = REPO_ROOT / "data" / "proxy_720p30.mp4"
CACHE_DIR = REPO_ROOT / "cache_dir"
CLIP_OUTPUT_DIR = REPO_ROOT / "outputs" / "validation_clips"

WINDOW_SECONDS = 15 #DEFAULT_WINDOW_SECONDS  # 8.0
STRIDE_SECONDS = 5 #DEFAULT_STRIDE_SECONDS  # 2.0
INDEX_DIR = REPO_ROOT / default_index_dir(VIDEO_PATH, WINDOW_SECONDS, STRIDE_SECONDS)

CLIP_CONTEXT_SECONDS = DEFAULT_CLIP_CONTEXT_SECONDS  # 2.0

# Opciones: "load", "build_if_missing", "compute_session".
EMBEDDING_MODE = "load"

BATCH_SIZE = DEFAULT_BATCH_SIZE  # 48
NUM_WORKERS = DEFAULT_NUM_WORKERS  # 4
TOP_K = 3

VIDEO_WEIGHT = DEFAULT_VIDEO_WEIGHT  # 0.5
AUDIO_WEIGHT = DEFAULT_AUDIO_WEIGHT  # 0.5

USE_HELLDIVERS_ADAPTER = True  # True
HELLDIVERS_ADAPTER_PATH = REPO_ROOT / "helldivers_adapter.pth"
ADAPTER_EPOCHS = 300  # Necesitamos muchos
ADAPTER_BATCH_SIZE = 4  
ADAPTER_NUM_WORKERS = 4  
ADAPTER_LEARNING_RATE = 5e-3 # y que sea agresivo

WHISPER_DATA_DIR = REPO_ROOT / "data" / "whisper"
WHISPER_DATA_DIR.mkdir(parents=True, exist_ok=True)

AUTO_LABEL_OUTPUT = WHISPER_DATA_DIR / "dataset_whisper.json"
AUTO_LABEL_WHISPER_MODEL = "openai/whisper-large-v3-turbo"  # openai/whisper-large-v3-turbo
AUTO_LABEL_KEEP_TOP_PCT = 0.15  # 0.15

CLEAN_DATASET_OUTPUT = WHISPER_DATA_DIR / "dataset_cleaned.json"

OCR_DATA_DIR = REPO_ROOT / "data" / "ocr"
OCR_DATA_DIR.mkdir(parents=True, exist_ok=True)

OCR_DATASET_OUTPUT = OCR_DATA_DIR / "dataset_ocr_raw.json"
OCR_WINDOW_SECONDS = 15.0  # 15.0
OCR_STRIDE_SECONDS = 5.0  # 5.0
OCR_MIN_CONFIDENCE = 0.35  # 0.35

OCR_CLEAN_DATASET_OUTPUT = OCR_DATA_DIR / "dataset_ocr_cleaned.json"
MASTER_DATASET_OUTPUT = REPO_ROOT / "data" / "master_dataset_raw.json"

MASTER_DATASET_FINAL_OUTPUT = REPO_ROOT / "data" / "master_dataset_final.json"
REWRITE_MODEL = "llama3.1" 

print(f"Repo root: {REPO_ROOT}")
print(f"Video: {VIDEO_PATH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Video/audio weights: {VIDEO_WEIGHT}/{AUDIO_WEIGHT}")
print(f"Use Helldivers adapter: {USE_HELLDIVERS_ADAPTER}")
print(f"Index dir: {INDEX_DIR}")
print(f"Embedding mode: {EMBEDDING_MODE}")
print(f"Window/stride: {WINDOW_SECONDS}s / {STRIDE_SECONDS}s")
print(f"Validation clip context: +/-{CLIP_CONTEXT_SECONDS}s")


Repo root: /home/ruben/Documents/AINE/aine-highlights
Video: /home/ruben/Documents/AINE/aine-highlights/data/proxy_720p30.mp4
Batch size: 48
DataLoader workers: 4
Video/audio weights: 0.5/0.5
Use Helldivers adapter: True
Index dir: /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5
Embedding mode: load
Window/stride: 15s / 5s
Validation clip context: +/-2.0s


## Dataset Whisper

Genera un dataset automático con la cascada volumen -> Whisper. El idioma se fuerza a español en la llamada al script.


In [3]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "auto_label_dataset.py"),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(AUTO_LABEL_OUTPUT),
        "--window-seconds",
        str(WINDOW_SECONDS),
        "--stride-seconds",
        str(STRIDE_SECONDS),
        "--keep-top-pct",
        str(AUTO_LABEL_KEEP_TOP_PCT),
        "--whisper-model",
        AUTO_LABEL_WHISPER_MODEL,
        "--language",
        "spanish",
    ],
    check=True,
)

print(f"Dataset Whisper guardado en: {AUTO_LABEL_OUTPUT}")

Loaded video=/home/ruben/Documents/AINE/aine-highlights/data/proxy_720p30.mp4, duration=6221.20s, fps=30.00, windows=1243, audio_sr=48000
Phase 1 kept 186/1243 windows (top 15%).


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Whisper labeling: 100%|██████████| 186/186 [01:24<00:00,  2.21it/s]


Saved 183 labels to /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_whisper.json
Dataset Whisper guardado en: /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_whisper.json


### Limpiar Dataset Whisper

Deduplica ventanas solapadas usando proximidad temporal y similitud de texto. El resultado limpio queda junto al dataset crudo.


In [4]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "clean_dataset.py"),
        "--input",
        str(AUTO_LABEL_OUTPUT),
        "--output",
        str(CLEAN_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset Whisper limpio guardado en: {CLEAN_DATASET_OUTPUT}")


Original windows: 183 -> Cleaned windows: 105
Saved cleaned dataset to /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_cleaned.json
Dataset Whisper limpio guardado en: /home/ruben/Documents/AINE/aine-highlights/data/whisper/dataset_cleaned.json


## Dataset OCR

Detecta eventos del HUD de Helldivers 2 con OCR sobre el frame central de cada ventana. El resultado queda en `data/ocr`.


In [6]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "auto_label_ocr.py"),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(OCR_DATASET_OUTPUT),
        "--window-seconds",
        str(OCR_WINDOW_SECONDS),
        "--stride-seconds",
        str(OCR_STRIDE_SECONDS),
        "--roi",
        "hud",
        "--min-confidence",
        str(OCR_MIN_CONFIDENCE),
    ],
    check=True,
)

print(f"Dataset OCR guardado en: {OCR_DATASET_OUTPUT}")


OCR labeling: 100%|██████████| 1243/1243 [00:00<00:00, 314313.61it/s]


Saved 301 OCR labels to /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_raw.json
Dataset OCR guardado en: /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_raw.json


### Limpiar Dataset OCR

Deduplica las detecciones OCR cercanas en el tiempo usando el mismo limpiador de texto que el dataset Whisper.


In [7]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "clean_dataset.py"),
        "--input",
        str(OCR_DATASET_OUTPUT),
        "--output",
        str(OCR_CLEAN_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset OCR limpio guardado en: {OCR_CLEAN_DATASET_OUTPUT}")


Original windows: 301 -> Cleaned windows: 69
Saved cleaned dataset to /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_cleaned.json
Dataset OCR limpio guardado en: /home/ruben/Documents/AINE/aine-highlights/data/ocr/dataset_ocr_cleaned.json


## Unir Datasets

Combina el dataset limpio de Whisper con el dataset limpio de OCR en un único dataset maestro.


In [8]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "merge_datasets.py"),
        "--whisper",
        str(CLEAN_DATASET_OUTPUT),
        "--ocr",
        str(OCR_CLEAN_DATASET_OUTPUT),
        "--output",
        str(MASTER_DATASET_OUTPUT),
    ],
    check=True,
)

print(f"Dataset maestro guardado en: {MASTER_DATASET_OUTPUT}")


Whisper: 105 + OCR: 69 -> Master: 165
Saved master dataset to /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_raw.json
Dataset maestro guardado en: /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_raw.json


## Reescribir Dataset Maestro

Usa un LLM para convertir las etiquetas crudas en descripciones objetivas en inglés listas para entrenamiento.


In [ ]:
subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "rewrite_dataset.py"),
        "--input",
        str(MASTER_DATASET_OUTPUT),
        "--output",
        str(MASTER_DATASET_FINAL_OUTPUT),
        "--model",
        REWRITE_MODEL,
    ],
    check=True,
)

print(f"Dataset maestro final guardado en: {MASTER_DATASET_FINAL_OUTPUT}")


Rewriting labels:  99%|█████████▉| 164/165 [01:07<00:00,  4.42it/s]

Saved final dataset to /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_final.json
Dataset maestro final guardado en: /home/ruben/Documents/AINE/aine-highlights/data/master_dataset_final.json


Rewriting labels: 100%|██████████| 165/165 [01:07<00:00,  2.44it/s]


## Entrenar Adapter Helldivers

Entrena un linear probe sobre embeddings de vídeo congelados de LanguageBind y guarda solo los pesos del adapter.


In [35]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "train_linear_probe.py"),
        "--dataset",
        str(MASTER_DATASET_FINAL_OUTPUT),
        "--video",
        str(VIDEO_PATH),
        "--output",
        str(HELLDIVERS_ADAPTER_PATH),
        "--cache-dir",
        str(CACHE_DIR),
        "--epochs",
        str(ADAPTER_EPOCHS),
        "--batch-size",
        str(ADAPTER_BATCH_SIZE),
        "--num-workers",
        str(ADAPTER_NUM_WORKERS),
        "--lr",
        str(ADAPTER_LEARNING_RATE),
    ],
    check=True,
)

print(f"Adapter Helldivers guardado en: {HELLDIVERS_ADAPTER_PATH}")


/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
Epoch 300/300: 100%|██████████| 1/1 [00:00<00:00, 381.75it/s, loss=0.7181]


Saved adapter weights to /home/ruben/Documents/AINE/aine-highlights/helldivers_adapter.pth
Adapter Helldivers guardado en: /home/ruben/Documents/AINE/aine-highlights/helldivers_adapter.pth


## GPU Y Modelos

Exige CUDA, muestra la GPU disponible y carga una sola vez las ramas de vídeo/audio de LanguageBind y el tokenizer.


In [36]:
import torch

device = require_cuda()
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"Using GPU: {gpu_name}")
print(f"Compute capability: {capability[0]}.{capability[1]}")
print(f"PyTorch: {torch.__version__}, CUDA runtime: {torch.version.cuda}")


Using GPU: NVIDIA GeForce RTX 5070 Ti
Compute capability: 12.0
PyTorch: 2.11.0+cu128, CUDA runtime: 12.8


In [37]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(
    video_model,
    audio_model,
    video_processor,
    audio_processor,
    text_tokenizer,
) = load_languagebind_models(cache_dir=CACHE_DIR, device=device)

print("Loaded LanguageBind video/audio models on CUDA.")


Loaded LanguageBind video/audio models on CUDA.


## Embeddings

Elige de dónde salen los embeddings de escena. Cargar es lo más rápido; construir guarda un índice; calcular en sesión deja los embeddings solo en memoria.


In [38]:
if EMBEDDING_MODE not in {"load", "build_if_missing", "compute_session"}:
    raise ValueError('EMBEDDING_MODE must be one of: "load", "build_if_missing", "compute_session"')

if EMBEDDING_MODE == "load":
    if not (INDEX_DIR / "metadata.json").exists():
        raise FileNotFoundError(
            f"No pregenerated index found at {INDEX_DIR}. "
            "Switch EMBEDDING_MODE to 'build_if_missing' or 'compute_session'."
        )
    index = load_languagebind_index(INDEX_DIR)
    windows = index.windows
    scene_embeddings = index.scene_embeddings.to(device=device, dtype=torch.float32)
    duration_s = float(index.metadata["duration_s"])
    print(f"Loaded pregenerated index: {INDEX_DIR}")

elif EMBEDDING_MODE == "build_if_missing":
    if (INDEX_DIR / "metadata.json").exists():
        index = load_languagebind_index(INDEX_DIR)
        print(f"Loaded existing index: {INDEX_DIR}")
    else:
        print(f"No existing index at {INDEX_DIR}; building and saving it now.")
        index = build_languagebind_index(
            video_path=VIDEO_PATH,
            output_dir=INDEX_DIR,
            window_seconds=WINDOW_SECONDS,
            stride_seconds=STRIDE_SECONDS,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            cache_dir=CACHE_DIR,
            dtype="float16",
            overwrite=False,
        )
    windows = index.windows
    scene_embeddings = index.scene_embeddings.to(device=device, dtype=torch.float32)
    duration_s = float(index.metadata["duration_s"])

else:  # EMBEDDING_MODE == "compute_session"
    print("Computing embeddings in memory for this notebook session only.")
    video_reader, fps, duration_s = open_video_reader(VIDEO_PATH)
    audio_track = load_audio_track(VIDEO_PATH)
    windows = make_windows(
        duration_s=duration_s,
        window_seconds=WINDOW_SECONDS,
        stride_seconds=STRIDE_SECONDS,
    )
    with torch.no_grad():
        scene_embeddings = extract_scene_embeddings_batched(
            windows=windows,
            video_path=VIDEO_PATH,
            video_reader=video_reader,
            fps=fps,
            audio_track=audio_track,
            video_model=video_model,
            audio_model=audio_model,
            video_processor=video_processor,
            audio_processor=audio_processor,
            device=device,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            video_weight=VIDEO_WEIGHT,
            audio_weight=AUDIO_WEIGHT,
        )

print(
    f"Embeddings ready: mode={EMBEDDING_MODE}, windows={len(windows)}, "
    f"scene_embeddings={tuple(scene_embeddings.shape)}, device={scene_embeddings.device}"
)
print(f"First 3 windows: {windows[:3]}")


Loaded pregenerated index: /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5
Embeddings ready: mode=load, windows=1243, scene_embeddings=(1243, 768), device=cuda:0
First 3 windows: [WindowSpec(start_s=0.0, end_s=15.0), WindowSpec(start_s=5.0, end_s=20.0), WindowSpec(start_s=10.0, end_s=25.0)]


## Consulta De Texto

Codifica una consulta de texto y ordena los embeddings de escena disponibles. Cambia `QUERY_5` o la query usada abajo para iterar rápido.


In [46]:
import torch.nn.functional as F
from scripts.train_linear_probe import VideoTextAdapter

CONTEXT = "A gameplay clip of a first-person sci-fi video game where "
QUERY_1 = "the camera suddenly falls to the ground and the screen turns red while the gamer groans in defeat."
QUERY_2 = "a glowing beacon causes a huge explosion that accidentally blows up a teammate, while gamers laugh hysterically and one person shouts in anger."
QUERY_3 = "a massive bomb suddenly explodes sending a friendly soldier flying through the air, triggering loud laughter and an angry yell over voice chat."
QUERY_4 = "a player accidentally drops a giant airstrike on the team and says oops, followed by loud explosions, laughing, and angry screaming."
QUERY_5 = "reinforcements are available"
QUERY_6 = "a player is eliminated"
QUERY_7 = "a player uses a stratagem"
QUERY_8 = "A player is killed by friendly fire"
QUERY_9 = "A dropship lands to rescue the players"
QUERY_10 = "A barrage of airstrikes is called in, causing massive explosions and chaos on the battlefield"
QUERY_11 = "An objective is completed"
QUERY_12 = "An eagle airstrike is called in"
QUERY_13 = "a player panics and frantically asks for controls while being overwhelmed by a massive bug breach"
QUERY_14 = "a player expresses heavy frustration after being accidentally incinerated by the dropship engines during extraction"
QUERY_15 = "a player drops into the battlefield in a hellpod, returning to the chaotic action"
QUERY_16 = "the squad is in a critical situation with no reinforcements available"
QUERY_17 = "a player frantically runs away under heavy attack, creating extreme camera chaos"
QUERY_18 = "a teammate hilariously betrays the squad, resulting in an accidental death and loud laughter over voice chat"
QUERY_19 = "a player desperately calls in an orbital stratagem to stop a massive bug breach"
QUERY_20 = "a comrade is down in battle and the player realizes reinforcements are ready to be deployed"
QUERY_21 = "the squad completes a critical mission objective, bringing the operation closer to success"
QUERY_22 = "a player is eliminated in the middle of a massive firefight while teammates scream in panic"
QUERY_23 = "a player expresses shock and grief after realizing their own mistake led to a teammate's death"
QUERY_24 = "the squad successfully extracts, but not without chaotic screams and last-second friendly fire"

adapter = None
if USE_HELLDIVERS_ADAPTER and HELLDIVERS_ADAPTER_PATH.exists():
    embedding_dim = int(index.video_embeddings.shape[-1]) if "index" in globals() else int(scene_embeddings.shape[-1])
    adapter = VideoTextAdapter(embedding_dim=embedding_dim).to(device)
    adapter.load_state_dict(torch.load(HELLDIVERS_ADAPTER_PATH, map_location=device))
    adapter.eval()
    print(f"Adapter Helldivers cargado: {HELLDIVERS_ADAPTER_PATH}")
elif USE_HELLDIVERS_ADAPTER:
    print(f"Adapter Helldivers no encontrado, usando embeddings base: {HELLDIVERS_ADAPTER_PATH}")

with torch.no_grad():
    text_embedding = extract_text_embedding(
        model=video_model,
        tokenizer=text_tokenizer,
        query=QUERY_10,
        device=device,
    )
    if "index" in globals() and hasattr(index, "video_embeddings") and hasattr(index, "audio_embeddings"):
        video_embeddings = index.video_embeddings.to(device=device, dtype=torch.float32)
        audio_embeddings = index.audio_embeddings.to(device=device, dtype=torch.float32)
        if adapter is not None:
            video_embeddings = F.normalize(adapter(video_embeddings), dim=-1)
        query_scene_embeddings = weighted_fuse_embeddings(
            video_embeddings,
            audio_embeddings,
            video_weight= 0.8, #VIDEO_WEIGHT
            audio_weight=0.2, #AUDIO_WEIGHT
        )
    else:
        query_scene_embeddings = scene_embeddings

    results = rank_windows(
        windows=windows,
        scene_embeddings=query_scene_embeddings,
        text_embedding=text_embedding,
        top_k=TOP_K,
    )


Adapter Helldivers cargado: /home/ruben/Documents/AINE/aine-highlights/helldivers_adapter.pth


## Exportar Resultados

Exporta las mejores ventanas como clips MP4 cortos y los muestra dentro del notebook para validarlos rápido.


In [47]:
from IPython.display import Video, display

exported_results = export_result_clips(
    video_path=VIDEO_PATH,
    results=results,
    output_dir=CLIP_OUTPUT_DIR,
    query=QUERY_10,
    context_seconds=CLIP_CONTEXT_SECONDS,
    max_duration_s=duration_s,
)

print_results(exported_results)

for result in exported_results:
    print(f"\nRank {result.rank}: {result.start_s:.2f}s - {result.end_s:.2f}s | score={result.score:.4f}")
    display(Video(str(result.clip_path), embed=False, html_attributes="controls preload='metadata'"))



Top 3 windows:
1. 5080.00s - 5095.00s score=0.2942 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_01_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_5080.00s_5095.00s_clip_5078.00s_5097.00s.mp4
2. 4965.00s - 4980.00s score=0.2647 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_02_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_4965.00s_4980.00s_clip_4963.00s_4982.00s.mp4
3. 5120.00s - 5135.00s score=0.2561 clip=/home/ruben/Documents/AINE/aine-highlights/outputs/validation_clips/rank_03_a_barrage_of_airstrikes_is_called_in__causing_massive_explosions_and_chaos_on_the_battlefield_hit_5120.00s_5135.00s_clip_5118.00s_5137.00s.mp4

Rank 1: 5080.00s - 5095.00s | score=0.2942



Rank 2: 4965.00s - 4980.00s | score=0.2647



Rank 3: 5120.00s - 5135.00s | score=0.2561
